# Erste Datenqualitätsprüfung

Dieses Notebook prüft die Qualität der sieben SAP-Exporte.

## 1. Daten laden

In [1]:
from pathlib import Path
import hashlib
import re
from decimal import Decimal
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string(index=False) if isinstance(value, pd.DataFrame) else value)

DATA_DIR = None
if DATA_DIR is None:
    candidates = [p / 'DIC_Arbeitsprobe_Testdaten_SAP_MM'
                  for p in [Path.cwd(), *Path.cwd().parents]]
    DATA_DIR = next((p for p in candidates if p.is_dir()), None)
if DATA_DIR is None or not Path(DATA_DIR).is_dir():
    raise FileNotFoundError('Set DATA_DIR to the directory containing the seven CSV files.')
DATA_DIR = Path(DATA_DIR)

spec = {
    'LFA1': ('LFA1_VENDOR_MASTER.csv', ['LIFNR']),
    'MARA': ('MARA_MATERIAL_MASTER.csv', ['MATNR']),
    'EKKO': ('EKKO_PO_HEADER.csv', ['MANDT', 'EBELN']),
    'EKPO': ('EKPO_PO_ITEM.csv', ['MANDT', 'EBELN', 'EBELP']),
    'EKET': ('EKET_SCHEDULE_LINES.csv', ['MANDT', 'EBELN', 'EBELP', 'ETENR']),
    'LIKP': ('LIKP_DELIVERY_HEADER.csv', ['MANDT', 'VBELN']),
    'LIPS': ('LIPS_DELIVERY_ITEM.csv', ['MANDT', 'VBELN', 'POSNR']),
}
raw, data, source_hashes = {}, {}, {}
for name, (filename, keys) in spec.items():
    path = DATA_DIR / filename
    source_hashes[filename] = hashlib.sha256(path.read_bytes()).hexdigest()
    frame = pd.read_csv(path, sep=';', encoding='utf-8-sig', dtype=str,
                        keep_default_na=False, na_filter=False, on_bad_lines='error')
    frame.insert(0, 'source_record', range(2, len(frame) + 2))
    raw[name] = frame
    data[name] = frame.assign(**{c: frame[c].str.strip()
                                for c in frame.columns if c != 'source_record'})

issues = {}
def show_issue(name, frame, limit=8):
    issues[name] = frame.copy()
    print(f'{name}: {len(frame)} Datensätze (maximal {limit} angezeigt)')
    display(frame.head(limit))


## 2. Tabellenübersicht und Beispieldaten

In [2]:
overview = pd.DataFrame([
    {'table': name, 'file': filename, 'rows': len(raw[name]),
     'columns': len(raw[name].columns) - 1, 'candidate_key': ', '.join(keys)}
    for name, (filename, keys) in spec.items()
])
display(overview)
for name, frame in raw.items():
    print(name)
    display(frame.head(3))


,table,file,rows,columns,candidate_key
0,LFA1,LFA1_VENDOR_MASTER.csv,15,6,LIFNR
1,MARA,MARA_MATERIAL_MASTER.csv,24,5,MATNR
2,EKKO,EKKO_PO_HEADER.csv,216,12,"MANDT, EBELN"
3,EKPO,EKPO_PO_ITEM.csv,468,12,"MANDT, EBELN, EBELP"
4,EKET,EKET_SCHEDULE_LINES.csv,820,7,"MANDT, EBELN, EBELP, ETENR"
5,LIKP,LIKP_DELIVERY_HEADER.csv,957,6,"MANDT, VBELN"
6,LIPS,LIPS_DELIVERY_ITEM.csv,963,9,"MANDT, VBELN, POSNR"


LFA1


,source_record,LIFNR,NAME1,LAND1,ORT01,STRAS,WAERS
0,2,0000200001,Nordstein Baustoffe GmbH,DE,Hamburg,Billbrookdeich 118,EUR
1,3,0000200002,Hanse Dämmtechnik AG,DE,Lübeck,Travemünder Allee 7,EUR
2,4,0000200003,Elbe Zement & Mörtel GmbH,DE,Stade,Am Industriehafen 3,EUR


MARA


,source_record,MATNR,MAKTX,MTART,MEINS,MATKL
0,2,000000005001,"Zement CEM II/B-S 32,5 R 25 kg",HAWA,SAC,B01
1,3,000000005002,"Kalksandstein KS 2DF 12-1,8",HAWA,ST,B02
2,4,000000005003,"Porenbetonstein PP2-0,35 625x240",HAWA,ST,B02


...

,source_record,MANDT,EBELN,EBELP,ETENR,EINDT,MENGE,WEMNG
0,2,300,4700001195,00040,0001,2025-08-07,99.000,82.022
1,3,300,4700001176,00010,0002,2026-01-18,121.987,121.987
2,4,300,4700001076,00040,0001,2025-05-12,201.668,201.668


LIKP


,source_record,MANDT,VBELN,LFART,LIFNR,WADAT_IST,WERKS
0,2,300,1900001001,EL,0000200006,2025-10-24,2300
1,3,300,1900001002,EL,0000200006,2025-11-06,2200
2,4,300,1900001003,EL,0000200006,2025-12-05,2200


LIPS


,source_record,MANDT,VBELN,POSNR,MATNR,LFIMG,MEINS,EBELN,EBELP,WERKS
0,2,300,1900001890,000010,000000005023,51.731,ST,4700001198,00010,2000
1,3,300,1900001340,000010,000000005015,53.450,ST,4700001080,00030,2300
2,4,300,1900001651,000010,000000005013,10.637,ROL,4700001144,00040,2200


## 3. Fehlende Werte und Leerzeichen

In [3]:
missing_rows = []
for name, frame in data.items():
    keys = spec[name][1]
    for col in frame.columns.drop('source_record'):
        blank = frame[col].eq('')
        spaces = raw[name][col].ne(frame[col])
        missing_rows.append({'table': name, 'column': col, 'key_column': col in keys,
                             'blank_rows': int(blank.sum()),
                             'blank_pct': round(100 * blank.mean(), 2),
                             'outer_whitespace_rows': int(spaces.sum())})
    show_issue(name + '_missing_key', frame.loc[frame[keys].eq('').any(axis=1)])
missing_profile = pd.DataFrame(missing_rows)
display(missing_profile)


LFA1_missing_key: 0 Datensätze (maximal 8 angezeigt)


,source_record,LIFNR,NAME1,LAND1,ORT01,STRAS,WAERS


MARA_missing_key: 0 Datensätze (maximal 8 angezeigt)


,source_record,MATNR,MAKTX,MTART,MEINS,MATKL


EKKO_missing_key: 0 Datensätze (maximal 8 angezeigt)


...

LIKP_missing_key: 0 Datensätze (maximal 8 angezeigt)


,source_record,MANDT,VBELN,LFART,LIFNR,WADAT_IST,WERKS


LIPS_missing_key: 0 Datensätze (maximal 8 angezeigt)


,source_record,MANDT,VBELN,POSNR,MATNR,LFIMG,MEINS,EBELN,EBELP,WERKS


,table,column,key_column,blank_rows,blank_pct,outer_whitespace_rows
0,LFA1,LIFNR,True,0,0.00,0
1,LFA1,NAME1,False,0,0.00,1
2,LFA1,LAND1,False,1,6.67,0
3,LFA1,ORT01,False,0,0.00,0
4,LFA1,STRAS,False,0,0.00,0
5,LFA1,WAERS,False,0,0.00,0
6,MARA,MATNR,True,0,0.00,0
7,MARA,MAKTX,False,0,0.00,0
8,MARA,MTART,False,0,0.00,0
9,MARA,MEINS,False,0,0.00,0


## 4. Duplikate und Schlüsselkonflikte

In [4]:
duplicate_rows = []
for name, frame in data.items():
    keys = spec[name][1]
    fields = list(frame.columns.drop('source_record'))
    valid = frame.loc[~frame[keys].eq('').any(axis=1)]
    counts = valid.groupby(keys, dropna=False).size()
    variants = valid.drop_duplicates(fields).groupby(keys, dropna=False).size()
    conflict_keys = variants[variants > 1].reset_index()[keys]
    conflicts = valid.merge(conflict_keys, on=keys, how='inner', validate='many_to_one')
    exact = raw[name].duplicated(fields, keep=False)
    duplicate_rows.append({
        'table': name, 'exact_extra_rows_raw': int(raw[name].duplicated(fields).sum()),
        'duplicate_key_groups': int((counts > 1).sum()),
        'conflicting_key_groups': len(conflict_keys)})
    show_issue(name + '_exact_duplicates', raw[name].loc[exact])
    show_issue(name + '_key_conflicts', conflicts)
duplicate_profile = pd.DataFrame(duplicate_rows)
display(duplicate_profile)


LFA1_exact_duplicates: 0 Datensätze (maximal 8 angezeigt)


,source_record,LIFNR,NAME1,LAND1,ORT01,STRAS,WAERS


LFA1_key_conflicts: 0 Datensätze (maximal 8 angezeigt)


,source_record,LIFNR,NAME1,LAND1,ORT01,STRAS,WAERS


MARA_exact_duplicates: 0 Datensätze (maximal 8 angezeigt)


...

LIPS_exact_duplicates: 0 Datensätze (maximal 8 angezeigt)


,source_record,MANDT,VBELN,POSNR,MATNR,LFIMG,MEINS,EBELN,EBELP,WERKS


LIPS_key_conflicts: 12 Datensätze (maximal 8 angezeigt)


,source_record,MANDT,VBELN,POSNR,MATNR,LFIMG,MEINS,EBELN,EBELP,WERKS
0,23,300,1900001404,000010,000000005023,11.159,ST,4700001092,00010,2100
1,81,300,1900001331,000010,000000005017,18.170,ST,4700001077,00020,2000
2,322,300,1900001429,000010,000000005013,59.000,ROL,4700001098,00010,2300
3,407,300,1900001404,000010,000000005023,12.275,ST,4700001092,00010,2100
4,497,300,1900001151,000010,000000005001,0.967,PAL,4700001041,00020,2000
5,503,300,1900001429,000010,000000005013,64.900,ROL,4700001098,00010,2300
6,655,300,1900001159,000010,000000005013,4.301,ROL,4700001043,00030,2300
7,668,300,1900001656,000010,000000005010,138.583,ST,4700001147,00020,2300


,table,exact_extra_rows_raw,duplicate_key_groups,conflicting_key_groups
0,LFA1,0,0,0
1,MARA,0,0,0
2,EKKO,0,6,6
3,EKPO,7,9,2
4,EKET,0,1,1
5,LIKP,0,0,0
6,LIPS,0,6,6


## 5. Datumswerte

In [5]:
date_columns = {'EKKO': ['BEDAT', 'AEDAT'], 'EKET': ['EINDT'], 'LIKP': ['WADAT_IST']}
parsed_dates, date_rows = {}, []
for name, columns in date_columns.items():
    for col in columns:
        values = data[name][col]
        shape = values.str.fullmatch(r'\d{4}-\d{2}-\d{2}')
        parsed = pd.to_datetime(values.where(shape), format='%Y-%m-%d', errors='coerce')
        parsed_dates[name, col] = parsed
        bad = values.ne('') & parsed.isna()
        date_rows.append({'table': name, 'column': col, 'missing': int(values.eq('').sum()),
                          'invalid_nonblank': int(bad.sum()), 'min': parsed.min(), 'max': parsed.max()})
        show_issue(name + '_' + col + '_invalid_date', data[name].loc[bad])
date_profile = pd.DataFrame(date_rows)
display(date_profile)
changed_before_created = parsed_dates['EKKO', 'AEDAT'] < parsed_dates['EKKO', 'BEDAT']
show_issue('EKKO_change_before_order_date', data['EKKO'].loc[changed_before_created])


EKKO_BEDAT_invalid_date: 0 Datensätze (maximal 8 angezeigt)


,source_record,MANDT,EBELN,BUKRS,BSTYP,BSART,LIFNR,EKORG,EKGRP,WAERS,BEDAT,AEDAT,ERNAM


EKKO_AEDAT_invalid_date: 0 Datensätze (maximal 8 angezeigt)


,source_record,MANDT,EBELN,BUKRS,BSTYP,BSART,LIFNR,EKORG,EKGRP,WAERS,BEDAT,AEDAT,ERNAM


EKET_EINDT_invalid_date: 0 Datensätze (maximal 8 angezeigt)


...

LIKP_WADAT_IST_invalid_date: 35 Datensätze (maximal 8 angezeigt)


,source_record,MANDT,VBELN,LFART,LIFNR,WADAT_IST,WERKS
26,28,300,1900001027,EL,0000200009,27.07.2025,2100
37,39,300,1900001038,EL,0000200015,06.07.2025,2300
98,100,300,1900001099,EL,0000200011,15.09.2025,2000
103,105,300,1900001104,EL,0000200007,31.07.2025,2100
122,124,300,1900001123,EL,0000200011,12.01.2026,2300
132,134,300,1900001133,EL,0000200002,20.06.2026,2200
136,138,300,1900001137,EL,0000200004,07.09.2025,2200
143,145,300,1900001144,EL,0000200010,17.12.2025,2300


,table,column,missing,invalid_nonblank,min,max
0,EKKO,BEDAT,0,0,2025-01-09,2026-06-30
1,EKKO,AEDAT,0,0,2025-01-09,2026-06-30
2,EKET,EINDT,0,0,2025-01-21,2026-09-06
3,LIKP,WADAT_IST,7,35,2025-01-20,2026-09-06


EKKO_change_before_order_date: 0 Datensätze (maximal 8 angezeigt)


,source_record,MANDT,EBELN,BUKRS,BSTYP,BSART,LIFNR,EKORG,EKGRP,WAERS,BEDAT,AEDAT,ERNAM


## 6. Mengen und Beträge

In [6]:
numeric_columns = {'EKPO': ['MENGE', 'NETPR', 'NETWR'],
                   'EKET': ['MENGE', 'WEMNG'], 'LIPS': ['LFIMG']}
numbers, numeric_rows = {}, []
for name, columns in numeric_columns.items():
    for col in columns:
        values = data[name][col]
        valid = values.str.fullmatch(r'[+-]?\d+(?:\.\d+)?')
        parsed = values.map(lambda value: Decimal(value) if re.fullmatch(r'[+-]?\d+(?:\.\d+)?', value) else None)
        numbers[name, col] = parsed
        negative = parsed.map(lambda value: value is not None and value < 0)
        zero = parsed.map(lambda value: value is not None and value == 0)
        invalid = values.ne('') & ~valid
        good = parsed.dropna()
        numeric_rows.append({'table': name, 'column': col, 'missing': int(values.eq('').sum()),
                             'invalid_nonblank': int(invalid.sum()), 'negative': int(negative.sum()),
                             'zero': int(zero.sum()), 'min': str(good.min()) if len(good) else None,
                             'max': str(good.max()) if len(good) else None})
        show_issue(name + '_' + col + '_invalid_number', data[name].loc[invalid])
        show_issue(name + '_' + col + '_negative', data[name].loc[negative])
numeric_profile = pd.DataFrame(numeric_rows)
display(numeric_profile)
over_received = pd.Series([a is not None and b is not None and a > b
                          for a, b in zip(numbers['EKET', 'WEMNG'], numbers['EKET', 'MENGE'])],
                         index=data['EKET'].index)
show_issue('EKET_received_above_scheduled_review', data['EKET'].loc[over_received])


EKPO_MENGE_invalid_number: 0 Datensätze (maximal 8 angezeigt)


,source_record,MANDT,EBELN,EBELP,MATNR,TXZ01,MENGE,MEINS,NETPR,NETWR,WERKS,MATKL,LOEKZ


EKPO_MENGE_negative: 0 Datensätze (maximal 8 angezeigt)


,source_record,MANDT,EBELN,EBELP,MATNR,TXZ01,MENGE,MEINS,NETPR,NETWR,WERKS,MATKL,LOEKZ


EKPO_NETPR_invalid_number: 22 Datensätze (maximal 8 angezeigt)


...

LIPS_LFIMG_negative: 0 Datensätze (maximal 8 angezeigt)


,source_record,MANDT,VBELN,POSNR,MATNR,LFIMG,MEINS,EBELN,EBELP,WERKS


,table,column,missing,invalid_nonblank,negative,zero,min,max
0,EKPO,MENGE,0,0,0,2,0.000,1183.000
1,EKPO,NETPR,0,22,0,0,9.80,489.00
2,EKPO,NETWR,0,22,0,2,0.00,485210.70
3,EKET,MENGE,0,0,3,0,-549.000,1183.000
4,EKET,WEMNG,0,0,0,62,0.000,1183.000
5,LIPS,LFIMG,0,0,0,0,0.189,1183.000


EKET_received_above_scheduled_review: 24 Datensätze (maximal 8 angezeigt)


,source_record,MANDT,EBELN,EBELP,ETENR,EINDT,MENGE,WEMNG
16,18,300,4700001068,00010,0001,2025-07-11,103.000,112.036
76,78,300,4700001055,00010,0001,2025-11-29,915.000,1033.762
194,196,300,4700001059,00030,0001,2025-02-26,805.000,923.634
221,223,300,4700001210,00090,0001,2025-05-04,-43.721,43.721
233,235,300,4700001100,00010,0001,2025-03-02,217.862,242.828
265,267,300,4700001058,00030,0002,2026-03-24,637.013,690.909
272,274,300,4700001069,00020,0001,2026-06-01,811.000,908.864
277,279,300,4700001016,00010,0001,2026-05-14,120.672,136.393


## 7. Tabellenbeziehungen

In [7]:
relations = [
    ('EKPO', 'EKKO', ['MANDT', 'EBELN']),
    ('EKET', 'EKPO', ['MANDT', 'EBELN', 'EBELP']),
    ('LIPS', 'EKPO', ['MANDT', 'EBELN', 'EBELP']),
    ('LIPS', 'LIKP', ['MANDT', 'VBELN']),
    ('EKKO', 'LFA1', ['LIFNR']), ('LIKP', 'LFA1', ['LIFNR']),
    ('EKPO', 'MARA', ['MATNR']), ('LIPS', 'MARA', ['MATNR']),
]
relation_rows = []
for child_name, parent_name, keys in relations:
    child, parent = data[child_name], data[parent_name]
    blank = child[keys].eq('').any(axis=1)
    parent_valid = parent.loc[~parent[keys].eq('').any(axis=1)]
    parent_counts = parent_valid.groupby(keys).size().rename('parent_rows').reset_index()
    joined = child.loc[~blank].merge(parent_counts, on=keys, how='left', validate='many_to_one')
    missing = joined.loc[joined['parent_rows'].isna()]
    ambiguous = joined.loc[joined['parent_rows'].gt(1)]
    label = child_name + '_to_' + parent_name
    relation_rows.append({'relationship': label, 'child_rows': len(child),
                          'blank_reference_rows': int(blank.sum()), 'missing_parent_rows': len(missing),
                          'ambiguous_parent_rows': len(ambiguous)})
    show_issue(label + '_missing_parent', missing)
    show_issue(label + '_ambiguous_parent', ambiguous)
relation_profile = pd.DataFrame(relation_rows)
display(relation_profile)


EKPO_to_EKKO_missing_parent: 0 Datensätze (maximal 8 angezeigt)


source_record,MANDT,EBELN,EBELP,MATNR,TXZ01,MENGE,MEINS,NETPR,NETWR,WERKS,MATKL,LOEKZ,parent_rows


EKPO_to_EKKO_ambiguous_parent: 13 Datensätze (maximal 8 angezeigt)


source_record,MANDT,EBELN,EBELP,MATNR,TXZ01,MENGE,MEINS,NETPR,NETWR,WERKS,MATKL,LOEKZ,parent_rows
37,300,4700001160,00020,000000005002,"Kalksandstein KS 2DF 12-1,8",586.000,ST,340.05,199269.30,2200,B02,,2
72,300,4700001072,00010,000000005010,Bewehrungsstahl B500B d12 6 m,29.000,ST,163.04,4728.16,2200,B06,,2
124,300,4700001059,00030,000000005004,Dämmplatte EPS 035 100 mm,805.000,M2,236.37,190277.85,2200,B03,,2
133,300,4700001133,00010,000000005023,Waschtisch-Set Keramik 60 cm,746.000,ST,327.72,244479.12,2200,B12,,2
145,300,4700001053,00010,000000005023,Waschtisch-Set Keramik 60 cm,621.000,ST,327.72,203514.12,2300,B12,,2
158,300,4700001160,00030,000000005001,"Zement CEM II/B-S 32,5 R 25 kg",1091.000,SAC,9.80,10691.80,2200,B01,,2
168,300,4700001160,00010,000000005001,"Zement CEM II/B-S 32,5 R 25 kg",267.000,SAC,9.80,2616.60,2100,B01,,2
199,300,4700001059,00010,000000005004,Dämmplatte EPS 035 100 mm,256.000,M2,236.37,60510.72,2200,B03,L,2


EKET_to_EKPO_missing_parent: 5 Datensätze (maximal 8 angezeigt)


...

LIPS_to_MARA_missing_parent: 25 Datensätze (maximal 8 angezeigt)


source_record,MANDT,VBELN,POSNR,MATNR,LFIMG,MEINS,EBELN,EBELP,WERKS,parent_rows
10,300,1900001383,000010,5016,164.443,ST,4700001087,00030,2300,NaN
27,300,1900001175,000010,5022,102.243,M2,4700001047,00040,2300,NaN
29,300,1900001053,000010,5018,43.713,EIM,4700001014,00020,2200,NaN
76,300,1900001477,000010,5009,301.653,SAC,4700001108,00030,2200,NaN
129,300,1900001680,000010,5016,36.289,ST,4700001152,00010,2300,NaN
179,300,1900001957,000010,5017,379.133,ST,4700009096,00020,2100,NaN
196,300,1900001784,000010,5003,290.413,ST,4700001172,00030,2200,NaN
221,300,1900001818,000010,5023,738.000,ST,4700001182,00010,2100,NaN


LIPS_to_MARA_ambiguous_parent: 0 Datensätze (maximal 8 angezeigt)


source_record,MANDT,VBELN,POSNR,MATNR,LFIMG,MEINS,EBELN,EBELP,WERKS,parent_rows


relationship,child_rows,blank_reference_rows,missing_parent_rows,ambiguous_parent_rows
EKPO_to_EKKO,468,0,0,13
EKET_to_EKPO,820,0,5,18
LIPS_to_EKPO,963,0,7,21
LIPS_to_LIKP,963,0,0,0
EKKO_to_LFA1,216,0,0,0
LIKP_to_LFA1,957,0,28,0
EKPO_to_MARA,468,0,0,0
LIPS_to_MARA,963,0,25,0


## 8. Einheiten, Materialien und Niederlassungen

In [8]:
comparison_rows = []
def compare_fields(child_name, parent_name, keys, fields):
    child, parent = data[child_name], data[parent_name]
    valid_parent = parent.loc[~parent[keys].eq('').any(axis=1)]
    unique_parent = valid_parent.loc[~valid_parent.duplicated(keys, keep=False)]
    right = unique_parent[keys + fields].rename(columns={c: 'parent_' + c for c in fields})
    merged = child.loc[~child[keys].eq('').any(axis=1)].merge(
        right, on=keys, how='inner', validate='many_to_one')
    for col in fields:
        comparable = merged[col].ne('') & merged['parent_' + col].ne('')
        mismatch = comparable & merged[col].ne(merged['parent_' + col])
        label = f'{child_name}_to_{parent_name}_{col}'
        comparison_rows.append({'check': label, 'child_rows': len(child),
                                'compared_rows': int(comparable.sum()),
                                'not_compared_rows': len(child) - int(comparable.sum()),
                                'mismatch_rows': int(mismatch.sum())})
        show_issue(label + '_mismatch', merged.loc[mismatch])

compare_fields('EKPO', 'MARA', ['MATNR'], ['MEINS'])
compare_fields('LIPS', 'MARA', ['MATNR'], ['MEINS'])
compare_fields('LIPS', 'EKPO', ['MANDT', 'EBELN', 'EBELP'], ['MEINS', 'MATNR', 'WERKS'])
compare_fields('LIPS', 'LIKP', ['MANDT', 'VBELN'], ['WERKS'])
comparison_profile = pd.DataFrame(comparison_rows)
display(comparison_profile)
for name, col in [('EKPO', 'MEINS'), ('LIPS', 'MEINS'), ('EKPO', 'LOEKZ'),
                  ('EKPO', 'WERKS'), ('LIKP', 'WERKS'), ('LIPS', 'WERKS')]:
    print(name, col)
    display(data[name][col].value_counts(dropna=False).rename_axis(col).reset_index(name='rows'))


EKPO_to_MARA_MEINS_mismatch: 0 Datensätze (maximal 8 angezeigt)


,source_record,MANDT,EBELN,EBELP,MATNR,TXZ01,MENGE,MEINS,NETPR,NETWR,WERKS,MATKL,LOEKZ,parent_MEINS


LIPS_to_MARA_MEINS_mismatch: 9 Datensätze (maximal 8 angezeigt)


,source_record,MANDT,VBELN,POSNR,MATNR,LFIMG,MEINS,EBELN,EBELP,WERKS,parent_MEINS
201,210,300,1900001038,000010,000000005001,11.604,PAL,4700001011,00010,2300,SAC
205,214,300,1900001152,000010,000000005001,2.966,PAL,4700001041,00020,2000,SAC
263,273,300,1900001040,000010,000000005001,0.189,PAL,4700001011,00010,2300,SAC
283,293,300,1900001030,000010,000000005001,1.590,PAL,4700001009,00020,2200,SAC
286,296,300,1900001039,000010,000000005001,6.311,PAL,4700001011,00010,2300,SAC
401,413,300,1900001153,000010,000000005001,2.021,PAL,4700001042,00010,2300,SAC
480,497,300,1900001151,000010,000000005001,0.967,PAL,4700001041,00020,2000,SAC
509,527,300,1900001154,000010,000000005001,3.308,PAL,4700001042,00010,2300,SAC


LIPS_to_EKPO_MEINS_mismatch: 9 Datensätze (maximal 8 angezeigt)


...

,WERKS,rows
0,2100,126
1,2200,126
2,2000,112
3,2300,104


LIKP WERKS


,WERKS,rows
0,2200,259
1,2100,251
2,2300,225
3,2000,222


LIPS WERKS


,WERKS,rows
0,2200,259
1,2100,252
2,2300,228
3,2000,224


## 9. Mehrfachzuordnungen beim Join

In [9]:
item_key = ['MANDT', 'EBELN', 'EBELP']
schedule_counts = data['EKET'].groupby(item_key).size().rename('schedule_rows')
delivery_counts = data['LIPS'].groupby(item_key).size().rename('delivery_rows')
fanout = schedule_counts.to_frame().join(delivery_counts, how='inner').reset_index()
fanout['naive_join_rows'] = fanout['schedule_rows'] * fanout['delivery_rows']
fanout = fanout.loc[(fanout['schedule_rows'] > 1) & (fanout['delivery_rows'] > 1)]
show_issue('schedule_delivery_many_to_many', fanout.sort_values('naive_join_rows', ascending=False))


schedule_delivery_many_to_many: 243 Datensätze (maximal 8 angezeigt)


MANDT,EBELN,EBELP,schedule_rows,delivery_rows,naive_join_rows
300,4700001095,00010,3,6,18
300,4700001153,00020,3,6,18
300,4700001052,00010,3,5,15
300,4700001064,00010,3,5,15
300,4700001099,00030,3,5,15
300,4700001090,00010,3,5,15
300,4700001092,00010,3,5,15
300,4700001086,00010,3,5,15


## 10. Ergebnisse

In [10]:
findings = pd.DataFrame([{'check': name, 'records': len(frame)}
                         for name, frame in issues.items() if len(frame)])
if len(findings):
    display(findings.sort_values(['records', 'check'], ascending=[False, True]).reset_index(drop=True))
else:
    print('Keine Auffälligkeiten gefunden.')


assert all(hashlib.sha256((DATA_DIR / filename).read_bytes()).hexdigest() == digest
           for filename, digest in source_hashes.items()), 'A source file changed during execution.'
print('Quelldateien unverändert.')


check,records
schedule_delivery_many_to_many,243
LIKP_WADAT_IST_invalid_date,35
LIKP_to_LFA1_missing_parent,28
LIPS_to_MARA_missing_parent,25
EKET_received_above_scheduled_review,24
EKPO_NETPR_invalid_number,22
EKPO_NETWR_invalid_number,22
LIPS_to_EKPO_MATNR_mismatch,22
LIPS_to_EKPO_ambiguous_parent,21
EKET_to_EKPO_ambiguous_parent,18


Quelldateien unverändert.
